In [ ]:
!pip install transformers torch

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

BASE = "/content/drive/MyDrive/MedXChAIn"
LocalModels = "/content/drive/MyDrive/MedXChAIn/LocalModels"

# Model mimarisini tekrar tanımla
class MedCPTClassifier(nn.Module):
    def __init__(self, encoder, num_labels):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_embedding)

# Encoder ve modeli oluştur
encoder = AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder")
NUM_LABELS = 797
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = []
for i in [1, 2, 3]:
    checkpoint = torch.load(f"{LocalModels}/hospital_{i}_weights.pt", map_location=device)

    # Infer num_labels from the checkpoint itself
    num_labels = checkpoint["classifier.3.weight"].shape[0]
    print(f"Hospital {i} — detected num_labels: {num_labels}")

    model = MedCPTClassifier(encoder, num_labels)
    model.load_state_dict(checkpoint)
    weights.append(model.state_dict())
    print(f"Hospital {i} weights uploaded ✓")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Hospital 1 — detected num_labels: 793
Hospital 1 weights uploaded ✓
Hospital 2 — detected num_labels: 793
Hospital 2 weights uploaded ✓
Hospital 3 — detected num_labels: 793
Hospital 3 weights uploaded ✓


In [ ]:
# 3 modelin ağırlıklarının ortalamasını al
global_weights = {}

for key in weights[0].keys():
    global_weights[key] = torch.stack(
        [weights[i][key].float() for i in range(3)]
    ).mean(dim=0)

print("FedAvg is completed ✓")

FedAvg is completed ✓


In [ ]:
# Infer from the federated weights dict
num_labels = global_weights["classifier.3.weight"].shape[0]

global_model = MedCPTClassifier(encoder, num_labels).to(device)
global_model.load_state_dict(global_weights)

torch.save(global_model.state_dict(), f"{BASE}/global_model_weights.pt")
print(f"Global model is done ({num_labels} labels) + global_model_weights.pt ✓")

Global model is done (793 labels) + global_model_weights.pt ✓


In [ ]:
import pandas as pd
import json
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder")
CSV_DIR = "/content/drive/MyDrive/MedXChAIn/Train&TestCSVs"

class SymptomDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts  = df["symptoms"].fillna("").tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoded["input_ids"].squeeze(),
            "attention_mask": encoded["attention_mask"].squeeze(),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

global_model.eval()

for i in [1, 2, 3]:
    test_df = pd.read_csv(f"{CSV_DIR}/test_{i}.csv")
    test_dataset = SymptomDataset(test_df, tokenizer)
    test_loader  = DataLoader(test_dataset, batch_size=32)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            outputs = global_model(input_ids, attention_mask)
            preds   = outputs.argmax(1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    print(f"Hospital {i} test set → Global Model Accuracy: {acc*100:.2f}%")

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Hospital 1 test set → Global Model Accuracy: 73.84%
Hospital 2 test set → Global Model Accuracy: 73.87%
Hospital 3 test set → Global Model Accuracy: 73.97%
